In [1]:
import seaborn as sns
titanic = sns.load_dataset("titanic")

In [2]:
titanic.shape

(891, 15)

In [3]:
titanic=titanic.drop(columns=["deck"])

In [4]:
titanic.shape

(891, 14)

Quoi : colonne deck supprimée
Pourquoi : 688 valeurs manquantes sur 891. si je supprimais les lignes je ferais perdre 688 passagers, et remplir les cases inventerait l’information pour les trois quarts des passagers et on se retrouvera avec trop de fausses données.
Lignes : les 891 car la colonne disparaît pour tout le monde aucun passager n’est perdu

In [5]:
titanic=titanic.dropna(subset=["embark_town"])

Quoi : les 2 lignes où embark_town est vide sont supprimées
Pourquoi : seulement 2 passagers sur 891 n'ont pas le port renseigné donc la perte est négligeable mais j'aurais pu les remplir avec Southampton vu que cest le port qui se repète le plus mais j'ai préféré ne pas inventer d’information.
Lignes : 2 (il reste 889 passagers)

In [6]:
titanic.groupby(["class","sex"])["age"].median()

class   sex   
First   female    35.0
        male      40.0
Second  female    28.0
        male      30.0
Third   female    21.5
        male      25.0
Name: age, dtype: float64

In [7]:
age_median_group = titanic.groupby(["class","sex"])["age"].transform("median")
titanic["age"]= titanic["age"].fillna(age_median_group)

Quoi : j’ai rempli les 177 âges manquants avec l’âge médian du groupe du passager (même classe et même sexe).
Pourquoi : il manquait l’âge de 177 passagers sur 889. Si j’avais supprimé ces lignes, j’aurais perdu trop de monde, et l’âge est trop important pour supprimer toute la colonne. J’ai donc choisi de remplir les cases vides. J’ai d’abord regardé l’âge médian de chaque groupe : il va de 21,5 ans (femmes de 3e classe) à 40 ans (hommes de 1ere classe). Les âges sont donc très différents selon la classe et le sexe. Si j’avais mis le même âge pour tout le monde il aurait été faux pour beaucoup de passagers.
Lignes : 177 (il n’y a plus aucun âge manquant)

In [8]:
titanic.isna().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [9]:
titanic.duplicated().sum()

np.int64(118)

In [10]:
titanic[titanic.duplicated(keep=False)].sort_values(["fare", "age"]).head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
277,0,2,male,30.0,0,0,0.00,S,Second,man,True,Southampton,no,True
413,0,2,male,30.0,0,0,0.00,S,Second,man,True,Southampton,no,True
466,0,2,male,30.0,0,0,0.00,S,Second,man,True,Southampton,no,True
481,0,2,male,30.0,0,0,0.00,S,Second,man,True,Southampton,no,True
674,0,2,male,30.0,0,0,0.00,S,Second,man,True,Southampton,no,True
732,0,2,male,30.0,0,0,0.00,S,Second,man,True,Southampton,no,True
263,0,1,male,40.0,0,0,0.00,S,First,man,True,Southampton,no,True
633,0,1,male,40.0,0,0,0.00,S,First,man,True,Southampton,no,True
815,0,1,male,40.0,0,0,0.00,S,First,man,True,Southampton,no,True
611,0,3,male,25.0,0,0,7.05,S,Third,man,True,Southampton,no,True


Quoi : j’ai trouvé 118 lignes identiques à une autre ligne et j’ai décidé de ne pas les supprimer
Pourquoi : le tableau n’a pas de colonne avec le nom ou un numéro de passager, donc je ne peux pas savoir si deux lignes identiques sont la même personne ou deux personnes différentes. En les regardant j’ai vu que ce sont surtout des hommes seuls partis de Southampton souvent avec un billet gratuit. En plus beaucoup de ces lignes sont devenues identiques car je leur ai donné le même âge a cause de l’étape précédente avec la médiane de leur groupe. Ce sont donc très probablement des personnes différentes qui se ressemblent.
Lignes : 0 (118 doublons repérés aucun supprimé)

In [11]:
#titanic[["survived", "alive", "pclass", "class", "embarked", "embark_town"]].head(10)
for colonne in ["survived", "alive", "pclass", "class", "embarked", "embark_town"]:
    print(colonne, ":", titanic[colonne].unique())

survived : [0 1]
alive : <StringArray>
['no', 'yes']
Length: 2, dtype: str
pclass : [3 1 2]
class : ['Third', 'First', 'Second']
Categories (3, str): ['First', 'Second', 'Third']
embarked : <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
embark_town : <StringArray>
['Southampton', 'Cherbourg', 'Queenstown']
Length: 3, dtype: str


In [12]:
titanic=titanic.drop(columns=["alive", "pclass", "embarked"])

In [13]:
titanic.shape

(889, 11)

Quoi : j’ai supprimé alive, pclass et embarked.
Pourquoi : ces colonnes disaient exactement la même chose alive = survived, pclass = class, embarked = embark_town. J’ai vérifié avec unique() qu’elles avaient bien les mêmes valeurs, juste écrites différemment. J’ai gardé survived, parce qu’avec des 0 et des 1, je peux calculer un taux de survie. J’ai gardé class et embark_town parce que "First"  et "Southampton" par exempl sont plus faciles à lire que "1" et "S".
Lignes : les 889 (3 colonnes en moins pour tout le monde et aucun passager perdu)

In [14]:
titanic["survived"] = titanic["survived"].astype(bool)

In [15]:
titanic["survived"].unique()

array([False,  True])

In [16]:
titanic["survived"].mean()

np.float64(0.38245219347581555)

In [17]:
titanic["sex"] = titanic["sex"].astype("category")
titanic["embark_town"] = titanic["embark_town"].astype("category")
titanic["who"] = titanic["who"].astype("category")

In [18]:
titanic.dtypes

survived           bool
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
class          category
who            category
adult_male         bool
embark_town    category
alone              bool
dtype: object

Quoi : j’ai changé survived en bool, et sex, who et embark_town en catégories.
Pourquoi : survived ne veut dire que oui ou non, donc vrai/faux est plus logique qu’un nombre. J’ai vérifié qu’on peut toujours calculer le taux de survie avec . Les trois autres colonnes n’ont que 2 ou 3 valeurs qui se répètent donc ce sont des catégories et pas du texte libre.
Lignes : les 889 (le type change pour tout le monde, les valeurs restent les mêmes)

In [19]:
titanic["fare"].describe()

count    889.000000
mean      32.096681
std       49.697504
min        0.000000
25%        7.895800
50%       14.454200
75%       31.000000
max      512.329200
Name: fare, dtype: float64

In [20]:
q1 = titanic["fare"].quantile(0.25)
q3 = titanic["fare"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart
tres_chers = titanic[titanic["fare"] > borne_haute]
len(tres_chers)

114

In [21]:
tres_chers["class"].value_counts()

class
First     102
Third       7
Second      5
Name: count, dtype: int64

Quoi : j’ai cherché les billets anormalement chers avec le code de la diapo 32 . J’en ai trouvé 114 et j’ai décidé de les garder.
Pourquoi : 102 de ces billets sont en 1ere classe donc c’est normal qu’ils coûtent cher. Les autres sont sûrement des familles qui partageaient un seul billet, ce qui fait monter le prix. Ce sont des prix élevés mais pas des erreurs. Si je les supprimais je perdrais presque la moitié de la 1ere classe et mes analyses seraient faussées.
Lignes : 0 (114 billets repérés aucun supprimé)

In [22]:
titanic.to_csv("../data/titanic_propre.csv", index=False)